# Three-model comparison (no cache, tunable data)

Same unified-metrics methodology as `unified_metrics_comparison.ipynb`, trimmed to **three models**:

1. **SANOS KF** (`personal/SANOS_KF/sanos_kf.py`, `run_sanos_kf`) — online EKF on the exact
   tridiagonal SANOS DLV² density, run **with and without smoothing** (`cfg.smoothing`; smoothing =
   paper §3.1 production map, each density node blurred by a lognormal of variance `v = η·V_ATM`).
2. **KF on SVI** (the "spline" surface KF — 5-param Gatheral SVI slice).
3. **KF single-strike (1 s batches)** — per-strike random-walk EKF, one batch update per 1 s window.

**No caching.** All data is loaded live from `MassiveClient` + the tickers CSV, so the data
knobs in the CONFIG cell (`MIN_TRADES`, `TRADE_DATE`, puts/calls subset) take effect on a re-run.
Every model is scored in undiscounted **call** space (C/df) via `unified_metrics.evaluate`.

In [1]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    for d in (start, *start.parents):
        if (d / "pyproject.toml").exists() and (d / "quantdata").is_dir():
            return d
    raise RuntimeError(f"quant_data project root not found above {start}")


PROJECT_ROOT = _find_root(Path.cwd().resolve())
for p in (
    PROJECT_ROOT,
    PROJECT_ROOT / "personal" / "shared_functions",
    PROJECT_ROOT / "personal" / "SANOS_KF",
    PROJECT_ROOT / "personal" / "Kalman_filter_spline",
):
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

from datetime import date

import numpy as np
import pandas as pd
import polars as pl

import personal.Kalman_filter_spline.surface_model as sm
import personal.shared_functions.unified_metrics as um
from personal.Kalman_filter_spline.surface_kalman_filters import (
    build_iv_observations,
    run_spline_surface_kalman,
)
from personal.SANOS_KF.sanos_kf import SanosKFConfig, run_sanos_kf
from personal.shared_functions.kalman_filters import build_iv_surface
from quantdata.massive.client import MassiveClient

rows = []  # one metric dict per model -> um.comparison_table at the end

/Users/elie/Documents/GitHub/quant_data/quantdata/__init__.py:212: UserWarning: Using cache mode 'on' as specified in environment variable QUANTDATA_CACHE_MODE.


## CONFIG

Everything tunable lives here. Re-run from this cell down after changing a knob.

In [2]:
# ── data knobs (no caching — these take effect on re-run) ──
TRADE_DATE = date(2024, 1, 3)
TICKER = "SPY"
MIN_TRADES = 1  # min trade_count per option ticker to include
PUTS_ONLY, CALLS_ONLY = True, False
# Single subset switch reused by every model so the comparison is apples-to-apples.
EVAL_IS_CALL = CALLS_ONLY if (CALLS_ONLY or PUTS_ONLY) else None  # None => calls AND puts

# ── SANOS KF (run_sanos_kf) hyperparameters ──
# These match sweep_params.BASE + the best point of quick_sweep_free_sigma.py
# (ho_nse_med~0.45, mean~0.55 on SPY 2024-01-03). The grid params below are NOT the
# SanosKFConfig defaults (n_anchors=20, k_min=-0.20, k_max=0.15) — those defaults give
# a coarse, clipped strike grid and a much worse held-out fit, so set them explicitly.
SANOS_N_ANCHORS = 300  # cap on strike-aligned interior nodes (default 20 is far too coarse)
SANOS_K_MIN = -0.60  # log-moneyness grid lo — must cover the put wing (data ~ -0.56)
SANOS_K_MAX = 0.10  # log-moneyness grid hi (data ~ +0.07)
SANOS_BATCH_S = 1.0  # window width (seconds): one predict + one batch update per window
SANOS_Q_SCALE_DLV = 1e-3  # process noise scale for net DLV^2 per trading day
SANOS_P0_SCALE = 0.1  # initial covariance diagonal scale
SANOS_ELL = 5.0  # RBF correlation length in DOLLARS
SANOS_SIGMA_INIT = 0.04  # initial net DLV^2 (flat)
SANOS_CORRELATED = True
ETA = 0.06  # SANOS smoothness for the "with smoothing" run (v = ETA * V_ATM); best in sweep

# ── KF-on-SVI (spline) hyperparameters ──
SPLINE_Q_SCALE, SPLINE_P0_SCALE, SPLINE_BATCH_S = 1e-8, 1e-4, 1.0

# ── KF single-strike (1 s batch) hyperparameters ──
SINGLE_BATCH_S, SINGLE_Q_SCALE = 1.0, 4.1e-5

## Load raw data (shared by all models)

Fetched once from `MassiveClient` + the tickers CSV; `build_iv_observations` produces the IV-annotated quote/trade frames used by the SANOS and spline models.

In [3]:
client = MassiveClient()
df_tickers = pl.read_csv(PROJECT_ROOT / "personal" / f"spy_options_tickers_{TRADE_DATE}.csv")
df_spy = client.fetch_trades(TICKER, TRADE_DATE)

# IV-annotated quotes/trades (puts AND calls; subset is applied per model below).
df_quotes, df_trades = build_iv_observations(df_tickers, TRADE_DATE, df_spy, min_trades=MIN_TRADES)
print(
    f"build_iv_observations: {df_quotes.height:,} quote rows, "
    f"{df_quotes['strike'].n_unique()} strikes"
)

Skipping O:SPY240105C00388000: no matched trades
Skipping O:SPY240105C00424000: no matched trades
Skipping O:SPY240105C00425000: no matched trades
Skipping O:SPY240105C00441000: no matched trades
Skipping O:SPY240105C00443000: no matched trades
Skipping O:SPY240105C00453000: no matched trades
Skipping O:SPY240105C00481000: 46 bad quotes
Skipping O:SPY240105C00482000: 64 bad quotes
Skipping O:SPY240105C00483000: 842 bad quotes
Skipping O:SPY240105C00484000: 2129 bad quotes
Skipping O:SPY240105C00485000: 3888 bad quotes
Skipping O:SPY240105C00486000: 4860 bad quotes
Skipping O:SPY240105C00487000: 1630 bad quotes
Skipping O:SPY240105C00488000: 136 bad quotes
Skipping O:SPY240105C00489000: 130 bad quotes
Skipping O:SPY240105C00490000: 218 bad quotes
Skipping O:SPY240105C00491000: 370 bad quotes
Skipping O:SPY240105C00492000: 341 bad quotes
Skipping O:SPY240105C00493000: 342 bad quotes
Skipping O:SPY240105C00494000: 290 bad quotes
Skipping O:SPY240105C00495000: 202 bad quotes
Skipping O:SPY

## 1. SANOS KF — with and without smoothing

`run_sanos_kf` filters the exact tridiagonal SANOS DLV² density online, one batch update per `SANOS_BATCH_S` window (held-out prior = previous window's posterior). The subset is selected via `cfg.puts_only`/`cfg.calls_only` (so the ATM-variance estimate for smoothing can still see both wings). We run it twice: `smoothing=False` (intrinsic) and `smoothing=True` (η·V_ATM).

In [4]:
def sanos_kf_eval_table(df_res, batch_s):
    """Standard eval table from run_sanos_kf output. sanos_kf_price[_prior] are
    F-normalized calls C/(F·df) → multiply by F for call space C/df. Market mid/spread
    are PCP-converted to the same C/df space. One row per (window, strike) so the
    per-snapshot Breeden-Litzenberger negative mass in um.evaluate is clean.
    (Same methodology as sweep_params._run_one, so numbers match the sweep.)"""
    q = df_res.to_pandas().dropna(subset=["sanos_kf_price", "sanos_kf_price_prior"])
    F = q["forward"].to_numpy(float)
    K = q["strike"].to_numpy(float)
    ic = q["is_call"].to_numpy(bool)
    dfac = q["discount_factor"].to_numpy(float) if "discount_factor" in q else np.ones(len(q))
    dfac = np.where(np.isfinite(dfac) & (dfac > 0), dfac, 1.0)
    to_call = lambda p: np.where(ic, p, p + F - K)  # native undiscounted → call space (PCP)
    bid = q["bid_price"].to_numpy(float)
    ask = q["ask_price"].to_numpy(float)
    out = pd.DataFrame(
        {
            "ts": (q["ts_ns"].astype("int64") // int(batch_s * 1e9)).to_numpy(),
            "strike": K,
            "forward": F,
            "mid_call": to_call(0.5 * (bid + ask) / dfac),
            "half_spread": np.maximum(0.5 * (ask - bid) / dfac, 1e-10),
            "prior_call": q["sanos_kf_price_prior"].to_numpy(float) * F,
            "post_call": q["sanos_kf_price"].to_numpy(float) * F,
        }
    )
    return out.drop_duplicates(subset=["ts", "strike"], keep="last")


def run_sanos(smoothing, name):
    cfg = SanosKFConfig(
        n_anchors=SANOS_N_ANCHORS,
        k_min=SANOS_K_MIN,
        k_max=SANOS_K_MAX,
        batch_seconds=SANOS_BATCH_S,
        smoothing=smoothing,
        eta=ETA,
        Q_scale_dlv=SANOS_Q_SCALE_DLV,
        P0_scale=SANOS_P0_SCALE,
        ell=SANOS_ELL,
        Sigma_init=SANOS_SIGMA_INIT,
        correlated=SANOS_CORRELATED,
        puts_only=PUTS_ONLY,
        calls_only=CALLS_ONLY,
    )
    df_res, _ = run_sanos_kf(df_quotes, df_trades, cfg)
    tbl = sanos_kf_eval_table(df_res, SANOS_BATCH_S)
    m = um.evaluate(tbl, model_name=name)
    rows.append(m)
    return m


m_sanos_ns = run_sanos(smoothing=False, name="SANOS KF (no smoothing)")
m_sanos_s = run_sanos(smoothing=True, name=f"SANOS KF (smoothing eta={ETA})")
pd.DataFrame([m_sanos_ns, m_sanos_s])

,model,ho_nse_mean,ho_nse_med,ho_nse_std,is_nse_mean,is_nse_med,is_nse_std,ho_rmse_mean,ho_rmse_med,ho_rmse_std,...,is_rmse_med,is_rmse_std,ho_out_pct_mean,ho_out_pct_med,is_out_pct_mean,is_out_pct_med,neg_mass_mean,neg_mass_med,arb_frac,n_snapshots
0,SANOS KF (no smoothing),0.543630,0.468508,0.375719,0.450723,0.380679,0.319554,0.030470,0.025519,0.027796,...,0.025169,0.027871,13.579189,11.111111,9.437347,4.545455,-4.690976e-07,0.0,0.001516,24119
1,SANOS KF (smoothing eta=0.06),0.522598,0.444375,0.390577,0.430691,0.354951,0.330905,0.032205,0.027317,0.030106,...,0.027193,0.030187,12.665048,9.090909,8.799226,0.000000,-4.744399e-07,0.0,0.001516,24119


## 2. KF on SVI (spline surface)

5-param Gatheral SVI slice, fit & evaluated in 1 s batches (one time-proportional predict + one batch update per window). `prior_price` = pre-update (held-out) price, `post_price` = posterior.

In [ ]:
df_quotes_iv = df_quotes
df_trades_iv = df_trades
if EVAL_IS_CALL is not None:  # same subset as the SANOS model
    df_quotes_iv = df_quotes_iv.filter(pl.col("is_call") == EVAL_IS_CALL)
    df_trades_iv = df_trades_iv.filter(pl.col("is_call") == EVAL_IS_CALL)

cfg_spline = sm.SurfaceModelConfig(
    model=sm.SVI_SLICE_MODEL,
    Q_scale=SPLINE_Q_SCALE,
    Q_structure="scaled_identity",
    P0_scale=SPLINE_P0_SCALE,
)

q_res, t_res, df_state = run_spline_surface_kalman(
    df_quotes_iv, df_trades_iv, cfg_spline, batch_seconds=SPLINE_BATCH_S
)
q = q_res.sort(["strike", "is_call", "ts_ns"]).to_pandas()
q["window"] = q["ts_ns"].astype("int64") // int(SPLINE_BATCH_S * 1e9)
snap = q.dropna(subset=["kalman_price_prior"])

df_spline = um.from_option_prices(
    ts=snap["window"],
    strike=snap["strike"],
    forward=snap["forward"],
    is_call=snap["is_call"],
    bid=snap["bid_price"],
    ask=snap["ask_price"],
    prior_price=snap["kalman_price_prior"],
    post_price=snap["kalman_price"],
    discount_factor=snap["discount_factor"] if "discount_factor" in snap else None,
)
# One row per (window, strike), keep last — same snapshot methodology as SANOS.
df_spline = df_spline.drop_duplicates(subset=["ts", "strike"], keep="last")

In [6]:
# Negative mass: reconstruct the SVI smile over a strike grid at sampled times.
state_pd = df_state.to_pandas().sort_values("ts_ns").reset_index(drop=True)
n_params = cfg_spline.n_params
theta_cols = [f"theta_{j}" for j in range(n_params)]
grid_K_nm = np.linspace(
    float(df_quotes_iv["strike"].min()), float(df_quotes_iv["strike"].max()), 41
)
n_snap = min(1000, len(state_pd))
state_samp = state_pd.iloc[np.linspace(0, len(state_pd) - 1, n_snap, dtype=int)].reset_index(
    drop=True
)
q_sorted = q.sort_values("ts_ns").reset_index(drop=True)
asof = np.clip(
    np.searchsorted(q_sorted["ts_ns"].values, state_samp["ts_ns"].values, "right") - 1,
    0,
    len(q_sorted) - 1,
)
F_vec, T_vec = q_sorted["forward"].values[asof], q_sorted["T"].values[asof]
thetas = state_samp[theta_cols].values
C_mat = np.full((n_snap, len(grid_K_nm)), np.nan)
for i in range(n_snap):
    th, F_i, T_i = thetas[i], F_vec[i], T_vec[i]
    if not (np.isfinite(th).all() and F_i > 0 and np.isfinite(F_i) and T_i > 0):
        continue
    ivs = np.array([cfg_spline.model.observe(th, np.log(k / F_i), T_i)[0] for k in grid_K_nm])
    C_mat[i] = um.bs_call_price(
        np.full(len(grid_K_nm), F_i), grid_K_nm, np.full(len(grid_K_nm), T_i), ivs
    )
valid = np.isfinite(C_mat).all(axis=1)
nm_spline = um.negative_mass_surface(grid_K_nm, C_mat[valid], F_vec[valid])

m_spline = um.evaluate(df_spline, model_name="KF on SVI (spline)", neg_mass_values=nm_spline)
rows.append(m_spline)
m_spline

{'model': 'KF on SVI (spline)',
 'ho_nse_mean': 1.067907389598195,
 'ho_nse_med': 0.8959656024100358,
 'ho_nse_std': 3.7234569664153416,
 'is_nse_mean': 0.8372149076140988,
 'is_nse_med': 0.7732094856249474,
 'is_nse_std': 2.3413634295590375,
 'ho_rmse_mean': 0.07893283779135206,
 'ho_rmse_med': 0.07411622992082005,
 'ho_rmse_std': 0.06931714307966591,
 'is_rmse_mean': 0.0775813148570055,
 'is_rmse_med': 0.07369708338374385,
 'is_rmse_std': 0.0657578618933924,
 'ho_out_pct_mean': 37.53085299931319,
 'ho_out_pct_med': 37.096774193548384,
 'is_out_pct_mean': 30.04812289988321,
 'is_out_pct_med': 28.767123287671232,
 'neg_mass_mean': -0.0016954029086944519,
 'neg_mass_med': 0.0,
 'arb_frac': 0.408,
 'n_snapshots': 24119}

## 3. KF single-strike (1 s batches)

Per-strike random-walk EKF assimilated in 1 s windows (`build_iv_surface(..., batch_seconds=1.0)`). The held-out one-step-ahead prior is the **previous window's** posterior (no in-window leakage).

In [ ]:
df_quotes_iv_b, df_trades_iv_b = build_iv_surface(
    df_tickers,
    TRADE_DATE,
    df_spy=df_spy,
    min_trades=MIN_TRADES,
    simple=False,
    batch_seconds=SINGLE_BATCH_S,
    q_scale=SINGLE_Q_SCALE,
    is_call_filter=EVAL_IS_CALL,
)
qb = df_quotes_iv_b.sort(["strike", "is_call", "ts_ns"]).to_pandas()
qb["window"] = qb["ts_ns"].astype("int64") // int(SINGLE_BATCH_S * 1e9)
# held-out prior = previous window's posterior price (representative = last quote in window)
_post_w = qb.groupby(["strike", "is_call", "window"], as_index=False).agg(
    _post_win=("kalman_price", "last")
)
_post_w["kalman_price_prior"] = _post_w.groupby(["strike", "is_call"])["_post_win"].shift(1)
qb = qb.merge(
    _post_w[["strike", "is_call", "window", "kalman_price_prior"]],
    on=["strike", "is_call", "window"],
    how="left",
)
qb = qb.dropna(subset=["kalman_price_prior"])
df_single = um.from_option_prices(
    ts=qb["window"],
    strike=qb["strike"],
    forward=qb["forward"],
    is_call=qb["is_call"],
    bid=qb["bid_price"],
    ask=qb["ask_price"],
    prior_price=qb["kalman_price_prior"],
    post_price=qb["kalman_price"],
    discount_factor=qb["discount_factor"] if "discount_factor" in qb else None,
)
# One row per (window, strike), keep last — same snapshot methodology as SANOS.
df_single = df_single.drop_duplicates(subset=["ts", "strike"], keep="last")

In [ ]:
# Negative mass: asof-join post_call across strikes at sampled times, then BL.
puts_mask = ~qb["is_call"]
ds = df_single[df_single["ts"].isin(qb.loc[puts_mask, "window"])].copy()
uK = np.sort(ds["strike"].unique())
n_snap_s = min(500, ds["ts"].nunique())
ts_int = ds["ts"].astype(np.int64).values
snap_ts = np.linspace(ts_int.min(), ts_int.max(), n_snap_s, dtype=np.int64)
groups = {K: g.sort_values("ts").reset_index(drop=True) for K, g in ds.groupby("strike")}
ref = groups[uK[len(uK) // 2]]
ref_asof = np.clip(
    np.searchsorted(ref["ts"].astype(np.int64).values, snap_ts, "right") - 1, 0, len(ref) - 1
)
F_arr = ref["forward"].values[ref_asof]
C_s = np.full((n_snap_s, len(uK)), np.nan)
for j, Kj in enumerate(uK):
    g = groups[Kj]
    idx = np.clip(
        np.searchsorted(g["ts"].astype(np.int64).values, snap_ts, "right") - 1, 0, len(g) - 1
    )
    C_s[:, j] = g["post_call"].values[idx]
valid_s = np.isfinite(C_s).all(axis=1) & np.isfinite(F_arr)
nm_single = um.negative_mass_surface(uK, C_s[valid_s], F_arr[valid_s])

m_single = um.evaluate(
    df_single,
    model_name=f"KF single-strike (1s batch, q={SINGLE_Q_SCALE})",
    neg_mass_values=nm_single,
)
rows.append(m_single)
m_single

## Comparison table

In [ ]:
cmp = um.comparison_table(rows)
cmp

,ho_nse_mean,ho_nse_med,ho_nse_std,is_nse_mean,is_nse_med,is_nse_std,ho_rmse_mean,ho_rmse_med,ho_rmse_std,is_rmse_mean,is_rmse_med,is_rmse_std,ho_out_pct_mean,ho_out_pct_med,is_out_pct_mean,is_out_pct_med,neg_mass_mean,neg_mass_med,arb_frac,n_snapshots
model,,,,,,,,,,,,,,,,,,,,
SANOS KF (no smoothing),0.584196,0.490502,0.478317,0.476052,0.382605,0.405495,0.009161,0.007573,0.010264,0.008373,0.006897,0.009813,16.040997,12.500000,11.156039,0.00000,-2.468373e-07,0.000000,0.000704,24066
SANOS KF (smoothing eta=0.06),0.557147,0.457436,0.495157,0.450577,0.349922,0.415405,0.009025,0.007119,0.010710,0.008255,0.006378,0.010236,14.855757,11.111111,10.343851,0.00000,-2.440854e-07,0.000000,0.000660,24066
KF on SVI (spline),0.758539,0.722087,0.737848,0.621095,0.620478,0.380709,0.033960,0.034192,0.024725,0.033211,0.033873,0.024301,26.322084,25.000000,19.515016,18.57775,-1.832281e-03,0.000000,0.456000,24066
"KF single-strike (1s batch, q=4.1e-05)",0.549616,0.208819,0.857275,0.179051,0.109554,0.218974,0.008570,0.003847,0.016616,0.003739,0.001574,0.012647,17.275782,0.000000,1.295512,0.00000,-2.353421e-01,-0.193195,0.992000,1169482


In [ ]:
cmp.sort_values("ho_nse_mean").to_csv(
    PROJECT_ROOT / "personal" / f"three_model_comparison_{TRADE_DATE}.csv", index=True
)